In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from src.datasets import get_data_loaders, get_labels
from src.train import train
from src.evaluate import evaluate
from src.models import make_model
from src.utils import set_seed, load_model

set_seed(42)

In [2]:
dataset_name = 'CIFAR10'
model_name = 'resnet18'
device = torch.device('mps' if torch.mps.device_count() > 0 else ('cuda' if torch.cuda.is_available() else 'cpu'))
batch_size = 128
epochs = 50
learning_rate = 0.001
patience = 5
mc_passes = 20
num_classes = 10

In [3]:
train_loader, val_loader, test_loader = get_data_loaders(
    name=dataset_name, batch_size=batch_size
)
model = make_model(model_name)
model = model.to(device)

In [5]:
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

In [6]:
checkpoint_path = f"{model_name}_{dataset_name}_best.pth"
train(model, train_loader, val_loader, optimizer, criterion,
      epochs=epochs, device=device, patience=patience, checkpoint_path=checkpoint_path)

model = load_model(model, checkpoint_path, device=device)

test_loss, test_accuracy, _ = evaluate(model, test_loader, criterion, device=device)

 74%|███████▍  | 37/50 [03:53<01:22,  6.32s/it, Epoch 38: Train Loss = 0.2983, Val Loss = 0.5143, Val Acc = 0.8430]


Early stopping triggered.


Evaluating: 100%|██████████| 79/79 [00:00<00:00, 149.57it/s]


In [7]:
test_loss, test_accuracy

(0.5290494210004807, 0.8277)

In [5]:
from src.losses import FairLoss

In [6]:
model = make_model(model_name)
model = model.to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = FairLoss()

In [11]:
checkpoint_path = f"{model_name}_{dataset_name}_best.pth"
train(model, train_loader, val_loader, optimizer, criterion,
      epochs=epochs, device=device, patience=patience, checkpoint_path=checkpoint_path)

model = load_model(model, checkpoint_path, device=device)

test_loss, test_accuracy, (all_predictions, all_targets) = evaluate(model, test_loader, criterion, device=device)

 22%|██▏       | 11/50 [01:16<04:30,  6.93s/it, Epoch 12: Train Loss = 0.3660, Val Loss = 0.5270, Val Acc = 0.8326]

Early stopping triggered.


In [10]:
test_loss, test_accuracy

(0.5425916210651398, 0.8239)

In [14]:
from src.metrics import Metric

ModuleNotFoundError: No module named 'sklearn'

In [12]:
m = Metric()
m.calculate_all_metrics(all_targets, all_predictions)

NameError: name 'Metric' is not defined